# FAIR-RANK — Part 2 Implementation & Empirical Demonstration Notebook

**Module:** COM713 Advanced Data Structures and Algorithms  
**Assessment:** Data Structure & Algorithm with Python — Part 2 (Individual Work, 30%)  
**Submission Deadline:** 16 August 2026  
**Dataset Source URL:** https://www.kaggle.com/datasets/snehaananthan/resume-dataset  

---  
### Overview
This notebook provides an end-to-end, step-by-step executable demonstration of the **FAIR-RANK** algorithmic resume screening system. FAIR-RANK addresses demographic bias and computational throughput limits on large-scale candidate pools.

The framework integrates four primary custom data structures:
1. **`SkillTrie`**: Token-level prefix tree for greedy longest-match phrase extraction.
2. **`SkillGraph`**: Adjacency list dictionary enforcing strict 1-hop synonym resolution.
3. **`InvertedIndex`**: Sub-linear candidate posting index.
4. **`TopKRanker`**: Bounded min-heap (`heapq`) achieving O(N log k) candidate ranking complexity.

In [1]:
# Cell 2: System Setup & Relative Imports
# Dataset Source URL: https://www.kaggle.com/datasets/snehaananthan/resume-dataset

import sys
import os
import csv
import json

base_dir = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(base_dir, 'src'))
sys.path.insert(0, base_dir)  # needed so `from benchmarks...` imports resolve later in the notebook

from fair_rank.job_description import JobDescription
from fair_rank.candidate import Candidate
from fair_rank.anonymizer import ResumeAnonymizer
from fair_rank.normalizer import TextNormalizer
from fair_rank.skill_trie import SkillTrie
from fair_rank.skill_graph import SkillGraph
from fair_rank.feature_extractor import FeatureExtractor
from fair_rank.inverted_index import InvertedIndex
from fair_rank.scorer import CandidateScorer
from fair_rank.top_k_ranker import TopKRanker
from fair_rank.fairness_auditor import FairnessAuditor
from fair_rank.explanation_generator import ExplanationGenerator
from fair_rank.screening_controller import ScreeningController

print('[+] FAIR-RANK modules successfully loaded from src/fair_rank/')

[+] FAIR-RANK modules successfully loaded from src/fair_rank/


In [2]:
# Cell 3: Load Controlled Vocabulary and Synonym Graph Data Assets

skills_dict_path = os.path.join(base_dir, 'data', 'skills_dictionary.json')
synonyms_path = os.path.join(base_dir, 'data', 'skills_synonyms.json')

with open(skills_dict_path, 'r', encoding='utf-8') as f:
    skills_vocab = json.load(f).get('skills', [])

with open(synonyms_path, 'r', encoding='utf-8') as f:
    synonyms_data = json.load(f)

print(f'Loaded {len(skills_vocab)} canonical skills into controlled vocabulary.')
print(f'Loaded {len(synonyms_data)} synonym mapping rules into adjacency graph.')
print('Sample synonyms:', dict(list(synonyms_data.items())[:5]))

Loaded 31 canonical skills into controlled vocabulary.
Loaded 18 synonym mapping rules into adjacency graph.
Sample synonyms: {'py': ['python'], 'python3': ['python'], 'js': ['javascript'], 'ts': ['typescript'], 'reactjs': ['react']}


In [3]:
# Cell 4: Demonstrate SkillTrie Construction and Greedy Phrase Extraction

trie = SkillTrie()
trie.build(skills_vocab)

toy_resume = 'Experienced engineer proficient in data structures, algorithms, python, and rest api.'
extracted_skills = trie.extract(toy_resume)

print('Original Resume Text:', toy_resume)
print('SkillTrie Extracted Skills:', sorted(list(extracted_skills)))
assert 'data structures' in extracted_skills, "Failed to extract phrase 'data structures'"

Original Resume Text: Experienced engineer proficient in data structures, algorithms, python, and rest api.
SkillTrie Extracted Skills: ['algorithms', 'data structures', 'python', 'rest api']


In [4]:
# Cell 5: Demonstrate SkillGraph Synonym Resolution

graph = SkillGraph(synonyms_data)
raw_skills = {'py', 'js', 'k8s', 'restful api'}
resolved_skills = graph.resolve_synonyms(raw_skills)

print('Raw Skill Terms:', raw_skills)
print('Resolved Canonical Skills:', sorted(list(resolved_skills)))
assert 'python' in resolved_skills, "Failed to map 'py' -> 'python'"
assert 'kubernetes' in resolved_skills, "Failed to map 'k8s' -> 'kubernetes'"

Raw Skill Terms: {'restful api', 'js', 'k8s', 'py'}
Resolved Canonical Skills: ['javascript', 'js', 'k8s', 'kubernetes', 'py', 'python', 'rest api', 'restful api']


In [5]:
# Cell 6: Demonstrate ResumeAnonymizer and TextNormalizer Preprocessing

sample_raw_resume = 'Alice Smith\nEmail: alice.smith@example.com | Phone: +1-555-0101 | DOB: 12/05/1990\nExperienced Senior Backend Engineer proficient in Python, SQL, REST API, Data Structures, AWS, Docker.'

masked_text = ResumeAnonymizer.mask(sample_raw_resume, name='Alice Smith')
normalized_text = TextNormalizer.normalize(masked_text)

print('=== ORIGINAL TEXT ===')
print(sample_raw_resume.strip())
print('\n=== MASKED TEXT (PII Stripped) ===')
print(masked_text.strip())
print('\n=== NORMALIZED TEXT ===')
print(normalized_text)

=== ORIGINAL TEXT ===
Alice Smith
Email: alice.smith@example.com | Phone: +1-555-0101 | DOB: 12/05/1990
Experienced Senior Backend Engineer proficient in Python, SQL, REST API, Data Structures, AWS, Docker.

=== MASKED TEXT (PII Stripped) ===
[NAME] [NAME]
Email: [EMAIL] | Phone: [PHONE] | [DOB]
Experienced Senior Backend Engineer proficient in Python, SQL, REST API, Data Structures, AWS, Docker.

=== NORMALIZED TEXT ===
name name email email phone phone dob experienced senior backend engineer proficient in python sql rest api data structures aws docker


In [6]:
# Cell 7: Load Dataset and Populate InvertedIndex

clean_data_path = os.path.join(base_dir, 'data', 'processed', 'resumes_clean.csv')
index = InvertedIndex()
candidates_dict = {}

with open(clean_data_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        cid = row['candidate_id']
        text = row['resume_text']
        name = row['full_name']
        
        masked = ResumeAnonymizer.mask(text, name=name)
        norm = TextNormalizer.normalize(masked)
        skills = graph.resolve_synonyms(trie.extract(norm))
        exp, edu, certs = FeatureExtractor.extract(norm)
        
        exp_val = float(row['experience_years']) if row.get('experience_years') else exp
        edu_val = row['education_level'] if row.get('education_level') else edu
        cert_val = {row['certifications'].strip().lower()} if row.get('certifications') and row['certifications'].strip() else certs
        
        cand = Candidate(
            candidate_id=cid,
            masked_text=masked,
            original_text=text,
            skills=skills,
            experience_years=exp_val,
            education_level=edu_val,
            certifications=cert_val
        )
        
        index.add(cand)
        candidates_dict[cid] = cand

print(f'Successfully indexed {len(candidates_dict)} candidate profiles into InvertedIndex.')
print('Sample index entry for python:', list(index.index.get('python', set()))[:5])

Successfully indexed 10 candidate profiles into InvertedIndex.
Sample index entry for python: ['C001', 'C009', 'C006', 'C005', 'C007']


In [7]:
# Cell 8: Instantiate JobDescription Specification

target_job = JobDescription(
    title='Senior Backend Engineer — Python & Cloud',
    required_skills={'python', 'sql', 'data structures', 'rest api'},
    preferred_skills={'aws', 'docker', 'kubernetes', 'pytest'},
    min_experience_years=4.0,
    required_education='Bachelor',
    required_certifications={'aws certified developer'},
    k=5,
    weights={
        'required_skills': 0.50,
        'preferred_skills': 0.20,
        'experience': 0.15,
        'education': 0.10,
        'certifications': 0.05,
    }
)

print('Job Specification:', target_job)
print('Component Weights:', target_job.weights)

Job Specification: <JobDescription title='Senior Backend Engineer — Python & Cloud' required_skills=4 k=5>
Component Weights: {'required_skills': 0.5, 'preferred_skills': 0.2, 'experience': 0.15, 'education': 0.1, 'certifications': 0.05}


In [8]:
# Cell 9: Evaluate CandidateScorer Component Score Breakdown

sample_candidate = candidates_dict['C001']
score_breakdown = CandidateScorer.score(sample_candidate, target_job)

print(f'=== SCORE BREAKDOWN FOR CANDIDATE {sample_candidate.candidate_id} ===')
for component, score in score_breakdown.items():
    print(f'- {component:<20}: {score * 100:.1f}%')

=== SCORE BREAKDOWN FOR CANDIDATE C001 ===
- required_skills     : 100.0%
- preferred_skills    : 100.0%
- experience          : 100.0%
- education           : 100.0%
- certifications      : 100.0%
- total               : 100.0%


In [9]:
# Cell 10: Candidate Retrieval & TopKRanker Bounded Min-Heap Shortlisting

relevant_ids = index.retrieve(target_job.required_skills, target_job.preferred_skills)
pool = [candidates_dict[cid] for cid in relevant_ids if cid in candidates_dict]

ranker = TopKRanker(k=target_job.k)
for cand in pool:
    bd = CandidateScorer.score(cand, target_job)
    cand.explanation = ExplanationGenerator.generate(cand, target_job, bd)
    ranker.push(cand, bd['total'])

shortlist = ranker.get_ranked()

print(f'=== TOP {target_job.k} RANKED SHORTLIST ===')
for rank, cand in enumerate(shortlist, start=1):
    print(f'Rank #{rank} | Candidate {cand.candidate_id} | Total Score: {cand.total_score*100:.1f}%')
    print(f'   Explanation: {cand.explanation}\n')

=== TOP 5 RANKED SHORTLIST ===
Rank #1 | Candidate C001 | Total Score: 100.0%
   Explanation: Overall Match Score: 100.0%. Required Skills (100% match): Matched [data structures, python, rest api, sql]. Preferred Skills (100% match): Matched [aws, docker, kubernetes, pytest]. Experience: 5.0 years (Required: 4.0 yrs). Education: Bachelor (Required: Bachelor).

Rank #2 | Candidate C006 | Total Score: 75.6%
   Explanation: Overall Match Score: 75.6%. Required Skills (75% match): Matched [python, rest api, sql]. Missing required skills: [data structures]. Preferred Skills (50% match): Matched [aws, docker]. Experience: 3.5 years (Required: 4.0 yrs). Education: Bachelor (Required: Bachelor).

Rank #3 | Candidate C008 | Total Score: 72.5%
   Explanation: Overall Match Score: 72.5%. Required Skills (75% match): Matched [data structures, rest api, sql]. Missing required skills: [python]. Preferred Skills (50% match): Matched [docker, pytest]. Experience: 4.0 years (Required: 4.0 yrs). Educati

In [10]:
# Cell 11: Execute FairnessAuditor Counterfactual Audit Suite

auditor = FairnessAuditor(trie, graph)
base_text = sample_candidate.original_text
audit_results = auditor.run_default_suite(target_job, base_text)

print('=== COUNTERFACTUAL FAIRNESS AUDIT REPORT ===')
for r in audit_results:
    print(f"- {r['test_type']:<35} | Score Diff: {r['score_difference']:.4f} | Passed: {r['passed']}")

=== COUNTERFACTUAL FAIRNESS AUDIT REPORT ===
- Name Bias (Male vs Female)          | Score Diff: 0.0000 | Passed: True
- Name Bias (Western vs Non-Western)  | Score Diff: 0.0000 | Passed: True
- Pronoun Bias                        | Score Diff: 0.0000 | Passed: True
- Email Domain Proxy Bias             | Score Diff: 0.0000 | Passed: True
- Formatting Whitespace Padding       | Score Diff: 0.0000 | Passed: True


In [11]:
# Cell 12: Run Empirical Efficiency Benchmark inline

from benchmarks.run_efficiency_benchmark import run_benchmark as run_eff_benchmark
run_eff_benchmark()

N Candidates    | Pool (Filtered)  | FAIR-RANK Time (s)   | Naive Time (s)     | Speedup   
--------------------------------------------------------------------------------------------
100             | 30               | 0.00031              | 0.00085            | 2.72      x


500             | 131              | 0.00123              | 0.00386            | 3.13      x


1000            | 286              | 0.00279              | 0.01414            | 5.07      x


5000            | 1466             | 0.01447              | 0.05001            | 3.46      x


10000           | 2914             | 0.04013              | 0.10251            | 2.55      x

[+] Results saved to /sessions/compassionate-peaceful-clarke/mnt/data structures and algo/project/fair-rank/benchmarks/results/efficiency_results.csv


[+] Empirical scaling plot saved to /sessions/compassionate-peaceful-clarke/mnt/data structures and algo/project/fair-rank/benchmarks/results/efficiency_plot.png


In [12]:
# Cell 13: Run Ranking Quality Evaluation inline

from benchmarks.run_ranking_quality_benchmark import run_quality_benchmark
run_quality_benchmark()

RANKING QUALITY BENCHMARK (k = 5)
Total Relevant Candidates in Population: 4
Relevant Candidates Retrieved in Shortlist: 4
Precision@5: 0.8000 (80.0%)
Recall@5:    1.0000 (100.0%)

[+] Quality benchmark metrics saved to /sessions/compassionate-peaceful-clarke/mnt/data structures and algo/project/fair-rank/benchmarks/results/ranking_quality_results.csv


## Cell 14: Discussion of Results & Research Questions (RQ1–RQ4)

### RQ1: Data Structure Efficiency
- SkillTrie enables O(L * d) token extraction per resume.
- SkillGraph provides O(1) synonym resolution without multi-hop drift.
- InvertedIndex enables sub-linear candidate retrieval.
- TopKRanker min-heap provides O(N log k) ranking versus naive O(N log N) sorting.

### RQ2: Candidate Ranking Quality
- Precision@5 = 80.0% and Recall@5 = 100.0%.

### RQ3: Algorithmic Fairness
- Counterfactual audit score difference = 0.0000 across identity proxies.

### RQ4: System Scalability
- Linear scaling across population sizes up to N = 10,000.

## Cell 15: Conclusion

Part 2 completes FAIR-RANK by delivering a working, tested, benchmarked Python framework.